In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

In [ ]:
df= pd.read_csv('data/train.csv')

In [ ]:
df.head(3)

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
0,0,0.0,No,6.0,4.0,No,15.0,5.0,Extrovert
1,1,1.0,No,7.0,3.0,No,10.0,8.0,Extrovert
2,2,6.0,Yes,1.0,0.0,NaN,3.0,0.0,Introvert


In [ ]:
df['Drained_after_socializing'].value_counts()

,count
Drained_after_socializing,
No,13313
Yes,4062


In [ ]:
df['Stage_fear'].value_counts()

,count
Stage_fear,
No,12609
Yes,4022


In [ ]:
df.isnull().sum()/df.shape[0]*100

,0
id,0.000000
Time_spent_Alone,6.424098
Stage_fear,10.219175
Social_event_attendance,6.370114
Going_outside,7.914057
Drained_after_socializing,6.202764
Friends_circle_size,5.689916
Post_frequency,6.823580
Personality,0.000000


In [ ]:
df= df.drop('id',axis=1)

In [ ]:
X = df.drop("Personality", axis=1)
y = df["Personality"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
numerical_cols = ['Time_spent_Alone', 'Social_event_attendance', 'Going_outside', 'Friends_circle_size', 'Post_frequency']
categorical_cols = ['Stage_fear', 'Drained_after_socializing']

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough'
)

In [ ]:
X_train=preprocessor.fit_transform(X_train)

In [ ]:
X_test=preprocessor.transform(X_test)

In [ ]:
pipe_logreg = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=10000))
])
pipe_svc = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC())
])
pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])
pipe_rf = Pipeline([
    ('model', RandomForestClassifier(random_state=42))
])


In [ ]:
model_hyperparameters = {

    pipe_logreg: {
        'model__C': [0.1, 1, 10]
    },

    pipe_svc: {
        'model__C': [0.1, 1, 10],
        'model__kernel': ['linear', 'rbf']
    },

    pipe_knn: {
        'model__n_neighbors': [3, 5, 7]
    },

    pipe_rf: {
        'model__n_estimators': [50, 100],
        'model__max_depth': [None, 10, 20]
    }
}

In [ ]:
from sklearn.model_selection import GridSearchCV
import pandas as pd

def ModelSelection(model_hyperparameters, X_train, y_train):

    results = []
    best_model = None
    best_score = 0

    for model, params in model_hyperparameters.items():

        grid = GridSearchCV(model, params, cv=5, scoring='accuracy')

        grid.fit(X_train, y_train)

        score = grid.best_score_

        results.append({
            "model": model,
            "best_score": score,
            "best_params": grid.best_params_
        })

        if score > best_score:
            best_score = score
            best_model = grid.best_estimator_

    results_df = pd.DataFrame(results)

    return results_df, best_model

In [ ]:
results_df, best_model = ModelSelection(model_hyperparameters, X_train, y_train)

print(results_df)

print("Best Model:", best_model)

                                               model  ...                                        best_params
0  (StandardScaler(), LogisticRegression(max_iter...  ...                                  {'model__C': 0.1}
1                          (StandardScaler(), SVC())  ...          {'model__C': 0.1, 'model__kernel': 'rbf'}
2         (StandardScaler(), KNeighborsClassifier())  ...                          {'model__n_neighbors': 7}
3          (RandomForestClassifier(random_state=42))  ...  {'model__max_depth': 10, 'model__n_estimators'...

[4 rows x 3 columns]
Best Model: Pipeline(steps=[('scaler', StandardScaler()), ('model', SVC(C=0.1))])


In [ ]:
y_pred = best_model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

   Extrovert       0.98      0.98      0.98      2740
   Introvert       0.94      0.95      0.94       965

    accuracy                           0.97      3705
   macro avg       0.96      0.96      0.96      3705
weighted avg       0.97      0.97      0.97      3705



In [ ]:
test_df = pd.read_csv("data/test.csv")

print(test_df.head())

      id  Time_spent_Alone  ... Friends_circle_size  Post_frequency
0  18524               3.0  ...                 6.0             NaN
1  18525               NaN  ...                 5.0             1.0
2  18526               3.0  ...                15.0             9.0
3  18527               3.0  ...                 5.0             6.0
4  18528               9.0  ...                 1.0             1.0

[5 rows x 8 columns]


In [ ]:

test_ids = test_df['id']
X_test_final = test_df.drop('id', axis=1)


X_test_processed = preprocessor.transform(X_test_final)


predictions = best_model.predict(X_test_processed)


output = pd.DataFrame({
    'id': test_ids,
    'Personality_Predicted': predictions
})

print(output.head())



      id Personality_Predicted
0  18524             Extrovert
1  18525             Introvert
2  18526             Extrovert
3  18527             Extrovert
4  18528             Introvert


In [ ]:
import joblib


joblib.dump(preprocessor, 'preprocessor.joblib')
joblib.dump(best_model, 'personality_model.joblib')

['personality_model.joblib']